# Figure 2 — Explore + InspectA standalone rebuild of the tool's **Explore** and **Inspect** modes for the workshoppaper, with the UI stripped away and only the maps, their visual encodings, and thetwo statistical charts kept.Each selected graph gets one row of four panels:| Explore | Inspect | node degree | edge length ||---|---|---|---|| all subgraphs, semantic colours | one subgraph, recoloured against the thresholds | degree distribution | length distribution |**Encodings are read from the tool, not re-invented.** Colours come from`frontend/nextjs/src/lib/colors.json`, the authoritative palette; node sizes, edgewidths, alphas, the mesh level ramp and the chart marks are the constants in`GNNExplorer.tsx` and `LeftPanel.tsx`. Graphs are loaded through the backend's own`services.graph`, so what is drawn is the same data the running tool would serve.**Threshold selection is the chip row at the top of the figure.** Every preset isdrawn once and hidden; a chip swaps which one is visible, so it costs nothing andstill works in an exported HTML file. Everything that changes what has to be loaded— which graph, which subgraph, where the crop sits — lives in the widgets in thesecond-to-last cell.Run the cells in order. Only **Cell 2 (configuration)** needs editing for adifferent figure; the widgets can drive the rest.

## 1 · Setup

In [ ]:
# ── Figure 2 · Explore + Inspect · setup ─────────────────────────────────────
import os, sys, json, math, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("/workspace/gnn-view")
os.environ["DATA_ROOT"] = str(ROOT / "backend" / "data")
sys.path.insert(0, str(ROOT / "backend" / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from services.graph import load_graph, get_graph_metrics, grid_spacing_deg
from services.bsf_converter import is_bsf_format, convert_bsf_to_neural_lam

# ── the UI palette, read from the single source of truth ─────────────────────
# frontend/nextjs/src/lib/colors.json is authoritative (see CLAUDE.md); never
# hardcode a hex here — every colour below is pulled from that file.
PAL = json.loads((ROOT / "frontend/nextjs/src/lib/colors.json").read_text())

NODE_GRID    = PAL["mapNodes"]["grid"]        # gold  · grid nodes  (squares)
NODE_MESH    = PAL["mapNodes"]["mesh"]        # mauve · mesh nodes  (circles)
EDGE_ENCODER = PAL["mapEdges"]["encoder"]     # crimson · G2M
EDGE_DECODER = PAL["mapEdges"]["decoder"]     # teal    · M2G
EDGE_MESH    = PAL["mapEdges"]["mesh"]        # mauve   · M2M / inter-level
VIOL_ABOVE   = PAL["violation"]["above"]      # over threshold
VIOL_BELOW   = PAL["violation"]["below"]      # under threshold
VIOL_NEUTRAL = PAL["violation"]["neutral"]    # within range
TEXT         = PAL["ui"]["text"]
TEXT_MUTED   = PAL["ui"]["textMuted"]
TEXT_DIM     = PAL["ui"]["textDim"]
UI_BG        = PAL["ui"]["background"]
CHART_GREY   = "#9ca3af"                      # GNNExplorer.tsx: unthresholded chart colour
AXIS_LINE    = "#e2e8f0"                      # GNNExplorer.tsx: axis + tick stroke

SANS = "SF Pro Display, SF Pro Text, -apple-system, Helvetica Neue, Arial, sans-serif"

def rgba(hex_color: str, alpha255: float) -> str:
    """UI colours carry 0-255 alphas; plotly wants rgba() with 0-1."""
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha255 / 255:.3f})"

def mix_white(hex_color: str, amount: float) -> str:
    """
    Fade a colour toward white. amount=1 leaves it alone, 0 erases it.

    Used for the basemap rather than a real alpha: the land, ocean and coastline
    fills stack on each other, so compositing them would darken every overlap.
    Mixing toward the white paper gives the same lightening with no seams.
    """
    h = hex_color.lstrip("#")
    a = min(1.0, max(0.0, amount))
    return "#" + "".join(f"{round(255 + (int(h[i:i + 2], 16) - 255) * a):02x}" for i in (0, 2, 4))

def grey_of(hex_color: str, alpha255: float) -> str:
    """GNNExplorer.hexToGrayRgba — luminance collapse used for de-emphasis."""
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    lum = round(0.299 * r + 0.587 * g + 0.114 * b)
    return f"rgba({lum},{lum},{lum},{alpha255 / 255:.3f})"

print("palette loaded ·", ROOT)

## 2 · ConfigurationEverything the figure exposes. Defaults are the two GraphCast graphs; the oceangraphs (`ocean_hierarchical`, `global_cluster_analysis`, `baltic_cluster`) work thesame way — they ship no position files, so they are converted once through thebackend's own converter and cached.

In [ ]:
# ── Figure 2 · configuration ────────────────────────────────────────────────
# Every knob the figure has. Nothing below this cell needs editing to change
# which graphs are shown, where the map is cropped, or how big the marks are.

# Where open_graph() looks up a bare folder name, in order.
GRAPH_SEARCH_DIRS = [
    ROOT / "notebooks" / "defect_graphs",
    ROOT / "backend" / "data" / "graphs",
    ROOT / "data" / "graphs",
    ROOT / "oceanography_graphs",
]
# Ocean graphs (Njord / baltic-sea-forecasting) ship no position files; they are
# converted to neural-lam format on first use and cached here.
GRAPH_CACHE_DIR = ROOT / "notebooks" / ".graph_cache"

# ── the two rows ────────────────────────────────────────────────────────────
# graph     folder name (or an absolute path) — anything open_graph() can resolve
# subgraph  registry key driving Inspect + both charts: "g2m" | "m2m" | "m2g"
#           | "m2m_0".. | "mesh_up_0".. | "mesh_down_0".. (hierarchical)
# label     row caption; None → the folder name
# crop      per-row override of CROP; None → the shared CROP below
ROWS = [
    dict(graph="global_multiscale",   subgraph="g2m", label="GraphCast · flat",         crop=None),
    dict(graph="global_hierarchical", subgraph="g2m", label="GraphCast · hierarchical", crop=None),
]

# ── map crop (shared) ───────────────────────────────────────────────────────
CROP = dict(
    center=(46.0, 8.0),    # (lat, lon) the panels are centred on
    height=18.0,           # degrees of latitude spanned, top to bottom
    aspect="auto",         # "auto" = fill the panel exactly; a number letterboxes
)

# ── what Explore draws ──────────────────────────────────────────────────────
# "all"  → every subgraph in the registry, exactly like the UI with all layers on
# or an explicit list, e.g. ["g2m", "m2m_0", "m2g"]
EXPLORE_SUBGRAPHS = "all"
EXPLORE_ARROWS    = True     # arrowheads on the directed (G2M / M2G) edges

# ── which chart the "nodes" panel shows ─────────────────────────────────────
NODE_DEGREE_SIDE = "src"     # "src" | "tgt"  — the UI shows both; the figure has room for one
NODE_CHART_STYLE = "lollipop"  # "lollipop" (UI encoding) | "bar"
LEN_BINS         = 15        # histogram bin count, UI default
LEN_CLUSTER_GAP  = 30.0      # km — UI clusterByGap() for discrete-length subgraphs

# ── threshold chips (Inspect + both charts) ─────────────────────────────────
# Rendered as a row of buttons at the top of the figure. Picking one recolours
# the Inspect panels and both charts; the maps and the axes do not move, because
# a threshold changes what is flagged, never what is there.
THRESHOLD_CHIPS = [
    ("none",   "no threshold"),
    ("1sigma", "mean ±1σ"),
    ("2sigma", "mean ±2σ"),
    ("3sigma", "mean ±3σ"),
    ("p5p95",  "p5–p95"),
    ("p1p99",  "p1–p99"),
]
THRESHOLD_PRESET = "2sigma"  # the chip selected when the figure opens; also the
                             # one baked into a static export. "custom" is valid
                             # here too and uses CUSTOM_THRESHOLDS below.
CUSTOM_THRESHOLDS = dict(    # used only when THRESHOLD_PRESET == "custom"
    src_deg=(None, None),    # (lo, hi); None → that side of the metric's full range
    tgt_deg=(None, None),
    length=(None, None),
)

# ── mark geometry ───────────────────────────────────────────────────────────
# Node size follows the UI rule: a node is a fixed fraction of one grid cell, so
# a dense ocean grid does not turn into a solid sheet (GNNExplorer NODE_SIZE_M).
NODE_SIZE_M          = 60_000     # UI constant, metres
REFERENCE_SPACING_DEG = 1.5       # ERA5 1.5° grid the constant was tuned for
NODE_SIZE_SCALE      = 1.0        # global multiplier if the figure wants bigger/smaller marks
NODE_SIZE_MIN_PX     = 1.6        # floor so sparse crops stay visible
NODE_SIZE_MAX_PX     = 13.0
MESH_SIZE_SCALE      = 2 / math.sqrt(math.pi)   # UI: equal-area circle vs square
MESH_NODE_COARSE_BOOST, MESH_NODE_FINE_BOOST = 3.5, 0.8   # LeftPanel.tsx
MESH_STROKE_COARSE,     MESH_STROKE_FINE     = 3.5, 0.8   # LeftPanel.tsx
NODE_BORDER_FRAC     = 1 / 24     # UI NODE_BORDER_DIVISOR — outline as a fraction of node size
NODE_BORDER_MIN_PX   = 0.4        # print floor; below this the outline vanishes
NODE_ICON_ALPHA         = 220     # UI constant (0-255) — node fill + outline in Explore
NODE_VIOL_ALPHA         = 230     # UI — flagged nodes in Inspect
NODE_VIOL_NEUTRAL_ALPHA = 180     # UI — within-range nodes in Inspect
EDGE_WIDTH_SCALE     = 0.55       # UI edge widths are CSS px; scale them for print
EDGE_ALPHA_NORMAL    = 80         # UI constant (0-255)
EDGE_ALPHA_VIOL_NEUTRAL = 45      # UI constant — greyed edges in Inspect
EDGE_ALPHA_VIOL_FLAG = 230        # UI constant — flagged edges in Inspect
ARROW_SIZE_FRAC      = 0.7        # UI: arrow icon is 0.7 x node size

# ── figure geometry ─────────────────────────────────────────────────────────
FIG_WIDTH_PX   = 1500
PANEL_H_PX     = 300
COL_WIDTHS     = [1.0, 1.0, 0.72, 0.72]   # explore, inspect, nodes chart, edges chart
H_GAP, V_GAP   = 0.028, 0.13
MARGIN         = dict(l=8, r=8, t=86, b=46)   # top leaves room for the chip row
CHIP_ROW_Y_PX  = 44           # chip row height above the plot area, in pixels
FONT_S, FONT_XS = 11, 9
SHOW_LEGENDS   = True
LEGEND_SCALE   = 1.0

# ── basemap (desaturated, per the design brief) ─────────────────────────────
# BASEMAP_ALPHA fades the whole basemap toward the white page at once, so the
# graph stays the darkest thing in the panel. 1.0 is the palette below at full
# strength; 0.0 leaves a blank page. The colours themselves are the reference
# values — retune those to change the map's hue, this to change its weight.
BASEMAP_ALPHA = 0.55

GEO = dict(
    resolution=50,                 # 50m Natural Earth; 110 for a coarser/faster outline
    landcolor="#eceff2",
    oceancolor="#dde5ea",
    lakecolor="#dde5ea",
    coastlinecolor="#b7c2cc",
    coastlinewidth=0.6,
    countrycolor="#dfe5ea",
    countrywidth=0.4,
    showcountries=False,
    framecolor="#cbd5e1",
    framewidth=1.0,
)

## 3 · Graphs, cropping, thresholds`open_graph` accepts anything the tool accepts: neural-lam flat, neural-lamhierarchical, and ocean graphs in Njord / baltic-sea-forecasting format.One repair happens here that the backend does not do: some ocean cluster graphsstore `mesh_up` / `mesh_down` indices in a combined two-level space rather thanper level, which overruns the coarser level's coordinate table. `localize_registry`detects that (an index that actually exceeds its node count) and rebases it,printing which subgraphs it touched.

In [ ]:
# ── Figure 2 · graph loading, cropping, thresholds ──────────────────────────

MERC_MAX = 85.05113

def merc_y(lat):
    """Web-Mercator y in degrees — the projection the UI map uses."""
    lat = np.clip(np.asarray(lat, dtype=float), -MERC_MAX, MERC_MAX)
    return np.degrees(np.log(np.tan(np.pi / 4 + np.radians(lat) / 2)))

def merc_y_inv(y):
    return np.degrees(2 * np.arctan(np.exp(np.radians(np.asarray(y, dtype=float)))) - np.pi / 2)


# ── resolving and opening a graph folder ────────────────────────────────────

def resolve_graph_dir(name) -> Path:
    p = Path(name)
    if p.is_absolute() and p.exists():
        return p
    for base in GRAPH_SEARCH_DIRS:
        if (base / name).is_dir():
            return base / name
    raise FileNotFoundError(f"graph '{name}' not found under {[str(b) for b in GRAPH_SEARCH_DIRS]}")

def available_graphs() -> list[str]:
    """Every folder under the search dirs that looks like a graph."""
    seen, out = set(), []
    for base in GRAPH_SEARCH_DIRS:
        if not base.is_dir():
            continue
        for d in sorted(base.iterdir()):
            if not d.is_dir() or d.name.startswith((".", "__")) or d.name in seen:
                continue
            files = {f.name for f in d.iterdir() if f.is_file()}
            if "grid_lat_lon.pt" in files or is_bsf_format(files):
                seen.add(d.name)
                out.append(d.name)
    return out

def localize_registry(raw: dict) -> list[str]:
    """
    Rebase any subgraph whose edge index is not local to its own node set.

    Most graphs store mesh_up / mesh_down indices per level, which is what the
    registry assumes. Some ocean cluster graphs instead store them in a combined
    two-level space, so one row starts at the size of the finer level and runs
    past the end of the coarser level's coordinate table. That is detectable
    without guessing: a row is only rebased when it actually overruns its node
    count, and only if subtracting the row minimum brings it back into range.
    Returns the keys it repaired, so a surprising graph says so out loud.
    """
    fixed, counts = [], raw["meta"]["node_counts"]
    for key, entry in raw["registry"].items():
        ei, changed = entry["edge_index"], False
        rows = [ei[0].clone(), ei[1].clone()]
        for r, side in ((0, "src"), (1, "tgt")):
            n, row = counts[entry[side]], rows[r]
            if int(row.max()) < n:
                continue
            shifted = row - int(row.min())
            if int(shifted.max()) < n:
                rows[r], changed = shifted, True
            else:
                raise ValueError(
                    f"subgraph '{key}' {side} index reaches {int(row.max())} but "
                    f"'{entry[side]}' has only {n} nodes; rebasing does not fix it."
                )
        if changed:
            entry["edge_index"] = torch.stack(rows)
            fixed.append(key)
    return fixed


def open_graph(name, projection: dict | None = None) -> dict:
    """
    load_graph() for any graph the tool accepts.

    Neural-lam folders load directly. Ocean folders (Njord / baltic-sea-forecasting)
    carry no position files, so they go through the same converter the upload route
    uses; the result is cached under GRAPH_CACHE_DIR and reused.
    """
    src = resolve_graph_dir(name)
    files = {f.name for f in src.iterdir() if f.is_file()}
    if is_bsf_format(files):
        out = GRAPH_CACHE_DIR / src.name
        if not (out / "grid_lat_lon.pt").exists():
            out.mkdir(parents=True, exist_ok=True)
            print(f"converting ocean graph '{src.name}' → {out} (once)…")
            info = convert_bsf_to_neural_lam(src, out, projection=projection)
            print(f"  {info['format']} · {info['levels']} level(s) · "
                  f"{info['grid_nodes']:,} grid nodes · flavour {info['flavour']}")
        src = out
    raw = load_graph(src)
    if not raw.get("_localized"):
        fixed = localize_registry(raw)
        raw["_localized"] = True
        if fixed:
            print(f"'{name}': rebased combined-space edge indices for {fixed}")
    return raw


# ── cropping ────────────────────────────────────────────────────────────────

def crop_bounds(crop: dict) -> dict:
    """
    (lat, lon) window for one panel.

    The height is given in latitude degrees; the width follows from the panel's
    aspect *in Mercator space*, so a cell stays square exactly as it does in the UI.
    """
    lat0, lon0 = crop["center"]
    half_h = crop["height"] / 2
    lat_lo, lat_hi = np.clip([lat0 - half_h, lat0 + half_h], -MERC_MAX, MERC_MAX)
    half_w = (merc_y(lat_hi) - merc_y(lat_lo)) / 2 * crop["aspect"]
    return dict(lat=(float(lat_lo), float(lat_hi)),
                lon=(float(lon0 - half_w), float(lon0 + half_w)),
                center=(float(lat0), float(lon0)))

def _lon_in(lon, lo, hi):
    """bbox_torch._lon_mask — longitude compared modulo 360."""
    lon = np.asarray(lon, dtype=float)
    if hi - lo >= 360:
        return np.ones_like(lon, dtype=bool)
    shift = math.floor((lo + 180) / 360) * 360
    lo, hi = lo - shift, hi - shift
    if hi <= 180:
        return (lon >= lo) & (lon <= hi)
    return (lon >= lo) | (lon <= hi - 360)

def node_mask(lat_lon: np.ndarray, b: dict) -> np.ndarray:
    lat, lon = lat_lon[:, 0], lat_lon[:, 1]
    return (lat >= b["lat"][0]) & (lat <= b["lat"][1]) & _lon_in(lon, *b["lon"])


# ── one row's data ──────────────────────────────────────────────────────────

def node_sets(raw: dict) -> dict[str, np.ndarray]:
    """{node_type: [N,2] lat/lon} for every node type in the graph."""
    out = {"grid": raw["grid_lat_lon"].numpy()}
    mesh = raw["mesh_lat_lon"]
    if raw["meta"]["is_hierarchical"]:
        for i, m in enumerate(mesh):
            out[f"mesh_{i}"] = m.numpy()
    else:
        out["mesh"] = mesh.numpy()
    return out

def explore_keys(raw: dict) -> list[str]:
    keys = list(raw["registry"].keys())
    if EXPLORE_SUBGRAPHS == "all":
        return keys
    return [k for k in EXPLORE_SUBGRAPHS if k in keys]

def edge_payload(raw: dict, folder, key: str, b: dict, nodes: dict) -> dict:
    """Cropped edges for one subgraph, with endpoint coordinates and lengths (km)."""
    entry = raw["registry"][key]
    m     = get_graph_metrics(key, folder)
    ei    = entry["edge_index"].numpy()
    src_ll, tgt_ll = nodes[entry["src"]], nodes[entry["tgt"]]
    smask, tmask   = node_mask(src_ll, b), node_mask(tgt_ll, b)
    keep = smask[ei[0]] | tmask[ei[1]]          # bbox_torch: OR semantics
    ei   = ei[:, keep]
    return dict(key=key, entry=entry,
                src_lat=src_ll[ei[0], 0], src_lon=src_ll[ei[0], 1],
                tgt_lat=tgt_ll[ei[1], 0], tgt_lon=tgt_ll[ei[1], 1],
                length=m["lengths"].numpy()[keep])

def build_row(spec: dict) -> dict:
    """Everything one row of the figure needs, cropped and thresholded."""
    folder = resolve_graph_dir(spec["graph"])
    if is_bsf_format({f.name for f in folder.iterdir() if f.is_file()}):
        folder = GRAPH_CACHE_DIR / folder.name
    raw   = open_graph(spec["graph"])
    meta  = raw["meta"]
    key   = spec["subgraph"]
    if key not in raw["registry"]:
        raise KeyError(f"'{key}' not in {sorted(raw['registry'])}")
    entry = raw["registry"][key]
    nodes = node_sets(raw)
    b     = crop_bounds(spec.get("crop") or CROP)

    metrics = get_graph_metrics(key, folder)
    src_deg = metrics["src_deg"].numpy()
    tgt_deg = metrics["tgt_deg"].numpy()
    lengths = metrics["lengths"].numpy()

    masks = {t: node_mask(ll, b) for t, ll in nodes.items()}
    return dict(
        spec=spec, folder=folder, raw=raw, meta=meta, key=key, entry=entry,
        label=spec.get("label") or spec["graph"],
        nodes=nodes, masks=masks, bounds=b,
        src_deg=src_deg, tgt_deg=tgt_deg, lengths=lengths,
        explore_edges=[edge_payload(raw, folder, k, b, nodes) for k in explore_keys(raw)],
        inspect_edges=edge_payload(raw, folder, key, b, nodes),
        n_levels=meta["n_levels"],
        spacing=meta["grid_spacing_deg"],
    )


# ── thresholds ──────────────────────────────────────────────────────────────

def metric_bounds(arr: np.ndarray, integer: bool) -> tuple[float, float]:
    """GNNExplorer.degBoundsGlobal — the padded full range the sliders span."""
    if arr is None or not len(arr):
        return (0.0, 30.0)
    lo, hi = float(np.min(arr)), float(np.max(arr))
    pad = max(1.0, round(0.05 * (hi - lo)))
    lo, hi = max(0.0, lo - pad), hi + pad
    return (math.floor(lo), math.ceil(hi)) if integer else (lo, hi)

def preset_thresholds(arr: np.ndarray, preset: str, integer: bool) -> tuple[float, float]:
    """
    (lo, hi) for one metric. A node/edge is flagged when value <= lo or >= hi,
    which is exactly the UI's rule, so 'none' returns the padded full range and
    flags nothing — the state the tool starts in.
    """
    full = metric_bounds(arr, integer)
    if preset == "none" or arr is None or not len(arr):
        return full
    v = np.asarray(arr, dtype=float)
    if preset.endswith("sigma"):
        k = float(preset[:-5])
        lo, hi = v.mean() - k * v.std(), v.mean() + k * v.std()
    elif preset == "p5p95":
        lo, hi = np.percentile(v, [5, 95])
    elif preset == "p1p99":
        lo, hi = np.percentile(v, [1, 99])
    else:
        raise ValueError(f"unknown preset '{preset}'")
    if integer:
        lo, hi = math.floor(lo), math.ceil(hi)
    lo, hi = max(full[0], lo), min(full[1], hi)
    return (float(min(lo, hi)), float(hi))

def row_thresholds(row: dict, preset: str = None, custom: dict = None) -> dict:
    """Per-metric (lo, hi) for one row, plus the full ranges the charts span."""
    preset = preset or THRESHOLD_PRESET
    custom = custom or CUSTOM_THRESHOLDS
    out = {}
    for name, arr, integer in (("src_deg", row["src_deg"], True),
                               ("tgt_deg", row["tgt_deg"], True),
                               ("length",  row["lengths"], False)):
        full = metric_bounds(arr, integer)
        if preset == "custom":
            c_lo, c_hi = custom.get(name, (None, None))
            lo = full[0] if c_lo is None else float(c_lo)
            hi = full[1] if c_hi is None else float(c_hi)
        else:
            lo, hi = preset_thresholds(arr, preset, integer)
        out[name] = dict(lo=lo, hi=hi, full=full, active=(lo > full[0] or hi < full[1]))
    return out

def viol_color(v, lo, hi, active: bool, neutral: str = VIOL_NEUTRAL) -> str:
    if not active:
        return CHART_GREY
    if v <= lo:
        return VIOL_BELOW
    if v >= hi:
        return VIOL_ABOVE
    return neutral

## 4 · Map panels — Explore and Inspect

In [ ]:
# ── Figure 2 · map panels (Explore + Inspect) ───────────────────────────────
# Every constant used here comes from GNNExplorer.tsx / LeftPanel.tsx, so the
# panels carry the same encodings as the tool: gold squares for grid nodes,
# mauve circles for mesh nodes sized by level, crimson G2M / teal M2G / mauve
# M2M edges in Explore, and the blue-grey-red violation ramp in Inspect.

def node_level_boost(node_type: str, n_levels: int) -> float:
    """LeftPanel.nodeLevelBoost — coarse mesh levels draw larger."""
    if n_levels <= 1 or not node_type.startswith("mesh_"):
        return 1.0
    lvl = int(node_type.split("_")[1])
    return MESH_NODE_FINE_BOOST + (MESH_NODE_COARSE_BOOST - MESH_NODE_FINE_BOOST) * lvl / (n_levels - 1)

def stroke_for_level(lvl: int, n_levels: int) -> float:
    """LeftPanel.strokeForLevel — coarse mesh levels draw heavier."""
    if n_levels <= 1:
        return (MESH_STROKE_COARSE + MESH_STROKE_FINE) / 2
    inv = n_levels - 1 - lvl
    return MESH_STROKE_COARSE - (MESH_STROKE_COARSE - MESH_STROKE_FINE) * inv / (n_levels - 1)

def base_node_px(row: dict, panel_w_px: float) -> float:
    """
    Grid-node size in points, following the UI's world-unit rule.

    GNNExplorer sizes a node at NODE_SIZE_M scaled by the graph's own grid
    spacing, so a node always covers the same fraction of one grid cell. Here
    that world size is converted into the panel's pixels via the crop width,
    which is what makes a 0.25 deg ocean grid read like the UI instead of
    fusing into a solid sheet.
    """
    spacing  = row["spacing"] or REFERENCE_SPACING_DEG
    size_deg = (NODE_SIZE_M / 111_320) * (spacing / REFERENCE_SPACING_DEG)
    lon_span = row["bounds"]["lon"][1] - row["bounds"]["lon"][0]
    px = size_deg / lon_span * panel_w_px * NODE_SIZE_SCALE
    return float(np.clip(px, NODE_SIZE_MIN_PX, NODE_SIZE_MAX_PX))


def _segments(src_lat, src_lon, tgt_lat, tgt_lon):
    """lat/lon arrays with a None after every segment; antimeridian jumps dropped."""
    keep = np.abs(tgt_lon - src_lon) <= 180
    n = int(keep.sum())
    if n == 0:
        return [], []
    nan = np.full(n, np.nan)
    lon = np.ravel(np.column_stack([src_lon[keep], tgt_lon[keep], nan]))
    lat = np.ravel(np.column_stack([src_lat[keep], tgt_lat[keep], nan]))
    return lat, lon

def edge_trace(geo, payload, mask, color, width, name=None):
    lat, lon = _segments(payload["src_lat"][mask], payload["src_lon"][mask],
                         payload["tgt_lat"][mask], payload["tgt_lon"][mask])
    if len(lat) == 0:
        return None
    return go.Scattergeo(lat=lat, lon=lon, mode="lines", geo=geo, name=name or "",
                         line=dict(color=color, width=width),
                         hoverinfo="skip", showlegend=False)

def arrow_trace(geo, payload, mask, color, size_px):
    """
    Arrowheads at the target end of a directed subgraph.

    The UI draws these as rotated sprites; here they are rotated triangle
    markers. The rotation is the edge's bearing in Mercator space, which is the
    space the panel is drawn in, so an arrow lines up with its own edge.
    """
    if not EXPLORE_ARROWS or not mask.any():
        return None
    s_lat, s_lon = payload["src_lat"][mask], payload["src_lon"][mask]
    t_lat, t_lon = payload["tgt_lat"][mask], payload["tgt_lon"][mask]
    keep = np.abs(t_lon - s_lon) <= 180
    if not keep.any():
        return None
    s_lat, s_lon, t_lat, t_lon = s_lat[keep], s_lon[keep], t_lat[keep], t_lon[keep]
    theta = np.degrees(np.arctan2(merc_y(t_lat) - merc_y(s_lat), t_lon - s_lon))
    return go.Scattergeo(
        lat=t_lat, lon=t_lon, mode="markers", geo=geo,
        marker=dict(symbol="triangle-up", size=size_px, color=color,
                    angle=90 - theta, angleref="up", line=dict(width=0)),
        hoverinfo="skip", showlegend=False)

def node_trace(geo, lat_lon, mask, symbol, size_px, color, border_px):
    if not mask.any():
        return None
    ll = lat_lon[mask]
    return go.Scattergeo(
        lat=ll[:, 0], lon=ll[:, 1], mode="markers", geo=geo,
        marker=dict(symbol=symbol, size=size_px, color=color,
                    line=dict(color=rgba(TEXT, NODE_ICON_ALPHA), width=border_px)),
        hoverinfo="skip", showlegend=False)

def explore_traces(row: dict, geo: str, panel_w_px: float) -> list:
    """The Explore panel: every subgraph in its own semantic colour, all node types."""
    n_lv   = row["n_levels"]
    size   = base_node_px(row, panel_w_px)
    border = max(NODE_BORDER_MIN_PX, size * NODE_BORDER_FRAC)
    out    = []

    # Edge layer order matches the UI: G2M under, mesh in the middle, M2G on top.
    def add_edges(payload):
        key, entry = payload["key"], payload["entry"]
        all_mask = np.ones(len(payload["length"]), dtype=bool)
        if key == "g2m":
            color, w = EDGE_ENCODER, 1.8
        elif key == "m2g":
            color, w = EDGE_DECODER, 1.8
        elif key.startswith("m2m"):
            lvl = int(key.split("_")[1]) if "_" in key else 0
            color, w = EDGE_MESH, stroke_for_level(lvl, n_lv)
        else:  # mesh_up_i / mesh_down_i — the finer level index is the suffix
            lvl = int(key.split("_")[2])
            color, w = EDGE_MESH, stroke_for_level(lvl, n_lv)
        out.append(edge_trace(geo, payload, all_mask,
                              rgba(color, EDGE_ALPHA_NORMAL), w * EDGE_WIDTH_SCALE))
        if key in ("g2m", "m2g"):
            out.append(arrow_trace(geo, payload, all_mask,
                                   rgba(color, EDGE_ALPHA_NORMAL), size * ARROW_SIZE_FRAC))

    ordered = sorted(row["explore_edges"],
                     key=lambda p: {"g2m": 0, "m2g": 2}.get(p["key"], 1))
    for payload in ordered:
        add_edges(payload)

    for t, ll in row["nodes"].items():
        m = row["masks"][t]
        if t == "grid":
            out.append(node_trace(geo, ll, m, "square", size,
                                  rgba(NODE_GRID, NODE_ICON_ALPHA), border))
        else:
            boost = node_level_boost(t, n_lv)
            out.append(node_trace(geo, ll, m, "circle", size * MESH_SIZE_SCALE * boost,
                                  rgba(NODE_MESH, NODE_ICON_ALPHA), border * boost))
    return [t for t in out if t is not None]


def inspect_traces(row: dict, thr: dict, geo: str, panel_w_px: float) -> list:
    """
    The Inspect panel: one subgraph, recoloured against the thresholds.

    Edges are split by length and nodes by degree into the same three states the
    tool uses — under, within, over — and the within state is deliberately faint
    (alpha 45 for edges, 180 for nodes) so the flagged elements carry the panel.
    """
    entry   = row["entry"]
    n_lv    = row["n_levels"]
    size    = base_node_px(row, panel_w_px)
    border  = max(NODE_BORDER_MIN_PX, size * NODE_BORDER_FRAC)
    payload = row["inspect_edges"]
    key     = row["key"]
    lengths = payload["length"]
    lo, hi, active = thr["length"]["lo"], thr["length"]["hi"], thr["length"]["active"]
    out = []

    if key in ("g2m", "m2g"):
        base_w = 1.8
    elif key.startswith("m2m"):
        base_w = stroke_for_level(int(key.split("_")[1]) if "_" in key else 0, n_lv)
    else:
        base_w = stroke_for_level(int(key.split("_")[2]), n_lv)

    if not active:
        color = {"g2m": EDGE_ENCODER, "m2g": EDGE_DECODER}.get(key, EDGE_MESH)
        out.append(edge_trace(geo, payload, np.ones(len(lengths), bool),
                              rgba(color, EDGE_ALPHA_NORMAL), base_w * EDGE_WIDTH_SCALE))
    else:
        below = lengths <= lo
        above = lengths >= hi
        mid   = ~(below | above)
        # UI: within-range edges thin out to 0.55x so flagged ones stand proud.
        out.append(edge_trace(geo, payload, mid,
                              rgba(VIOL_NEUTRAL, EDGE_ALPHA_VIOL_NEUTRAL),
                              base_w * 0.55 * EDGE_WIDTH_SCALE))
        out.append(edge_trace(geo, payload, below,
                              rgba(VIOL_BELOW, EDGE_ALPHA_VIOL_FLAG), base_w * EDGE_WIDTH_SCALE))
        out.append(edge_trace(geo, payload, above,
                              rgba(VIOL_ABOVE, EDGE_ALPHA_VIOL_FLAG), base_w * EDGE_WIDTH_SCALE))

    # Only the active subgraph's own node types are on screen, as in the tool.
    shown = [entry["src"]] if entry["src"] == entry["tgt"] else [entry["src"], entry["tgt"]]
    for t in shown:
        ll, m = row["nodes"][t], row["masks"][t]
        deg   = row["src_deg"] if t == entry["src"] else row["tgt_deg"]
        side  = "src_deg" if t == entry["src"] else "tgt_deg"
        d_lo, d_hi, d_on = thr[side]["lo"], thr[side]["hi"], thr[side]["active"]
        symbol = "square" if t == "grid" else "circle"
        boost  = 1.0 if t == "grid" else node_level_boost(t, n_lv)
        s      = size if t == "grid" else size * MESH_SIZE_SCALE * boost
        b      = border if t == "grid" else border * boost
        if not d_on:
            base = NODE_GRID if t == "grid" else NODE_MESH
            out.append(node_trace(geo, ll, m, symbol, s, rgba(base, NODE_ICON_ALPHA), b))
            continue
        d = np.asarray(deg)
        groups = [
            (m & (d > d_lo) & (d < d_hi), rgba(VIOL_NEUTRAL, NODE_VIOL_NEUTRAL_ALPHA)),
            (m & (d <= d_lo),             rgba(VIOL_BELOW,   NODE_VIOL_ALPHA)),
            (m & (d >= d_hi),             rgba(VIOL_ABOVE,   NODE_VIOL_ALPHA)),
        ]
        for gmask, color in groups:
            out.append(node_trace(geo, ll, gmask, symbol, s, color, b))
    return [t for t in out if t is not None]


def geo_layout(row: dict) -> dict:
    """One geo subplot: Mercator, cropped, desaturated basemap."""
    b = row["bounds"]
    tint = lambda c: mix_white(c, BASEMAP_ALPHA)   # frame excluded — panel chrome, not map
    return dict(
        projection=dict(type="mercator"),
        lataxis=dict(range=list(b["lat"])),
        lonaxis=dict(range=list(b["lon"])),
        showland=True, showocean=True, showlakes=True, showcoastlines=True,
        showframe=True, showrivers=False,
        landcolor=tint(GEO["landcolor"]), oceancolor=tint(GEO["oceancolor"]),
        lakecolor=tint(GEO["lakecolor"]),
        coastlinecolor=tint(GEO["coastlinecolor"]), coastlinewidth=GEO["coastlinewidth"],
        showcountries=GEO["showcountries"], countrycolor=tint(GEO["countrycolor"]),
        countrywidth=GEO["countrywidth"], resolution=GEO["resolution"],
        framecolor=GEO["framecolor"], framewidth=GEO["framewidth"],
        bgcolor="rgba(0,0,0,0)",
    )

## 5 · The two statistical charts

In [ ]:
# ── Figure 2 · the two statistical charts ───────────────────────────────────
# Node degree keeps the tool's lollipop (a degree is an integer, so a bar chart
# would imply a continuum it does not have); edge length keeps the tool's
# histogram, except for subgraphs whose lengths fall on discrete levels — the
# UI clusters those and draws a lollipop instead, and so does this.

def cluster_by_gap(values: np.ndarray, gap_km: float = None) -> list[tuple[float, int]]:
    """GNNExplorer.clusterByGap — collapse near-identical lengths onto their median."""
    gap_km = LEN_CLUSTER_GAP if gap_km is None else gap_km
    if not len(values):
        return []
    v = np.sort(np.asarray(values, dtype=float))
    cuts = np.flatnonzero(np.diff(v) > gap_km) + 1
    return [(float(round(np.median(c))), int(len(c))) for c in np.split(v, cuts) if len(c)]

def value_counts(values: np.ndarray) -> list[tuple[float, int]]:
    v, n = np.unique(np.asarray(values), return_counts=True)
    return list(zip(v.astype(float), n.astype(int)))


def lollipop_traces(entries, lo, hi, active, axis: tuple[str, str]) -> list:
    """
    Stem-and-dot frequency plot, exactly the tool's encoding: stems at 1.5 px /
    0.8 opacity, hollow dots with a coloured 1.2 px ring, coloured by which side
    of the thresholds the value falls on.
    """
    xa, ya = axis
    if not entries:
        return []
    buckets: dict[str, list] = {}
    for v, n in entries:
        buckets.setdefault(viol_color(v, lo, hi, active), []).append((v, n))
    out = []
    for color, pts in buckets.items():
        xs = np.ravel([[v, v, np.nan] for v, _ in pts])
        ys = np.ravel([[0, n, np.nan] for _, n in pts])
        out.append(go.Scatter(x=xs, y=ys, mode="lines", xaxis=xa, yaxis=ya,
                              line=dict(color=color, width=1.5), opacity=0.8,
                              hoverinfo="skip", showlegend=False))
        out.append(go.Scatter(x=[v for v, _ in pts], y=[n for _, n in pts], mode="markers",
                              xaxis=xa, yaxis=ya, opacity=0.9,
                              marker=dict(size=5, color="white",
                                          line=dict(color=color, width=1.2)),
                              customdata=[[v, n] for v, n in pts],
                              hovertemplate="%{customdata[0]:g} · n=%{customdata[1]:,}<extra></extra>",
                              showlegend=False))
    return out


def histogram_traces(values, domain, lo, hi, active, n_bins, axis) -> list:
    """Equal-width bins over the metric's full padded range, coloured per bin midpoint."""
    xa, ya = axis
    values = np.asarray(values, dtype=float)
    if not len(values):
        return []
    edges = np.linspace(domain[0], domain[1], max(2, n_bins) + 1)
    counts, _ = np.histogram(np.clip(values, domain[0], domain[1]), bins=edges)
    mids  = (edges[:-1] + edges[1:]) / 2
    width = (edges[1] - edges[0]) * 0.94
    colors = [viol_color(m, lo, hi, active) for m in mids]
    return [go.Bar(x=mids, y=counts, width=width, xaxis=xa, yaxis=ya,
                   marker=dict(color=colors, line=dict(width=0)), opacity=0.75,
                   hovertemplate="%{x:.0f} · n=%{y:,}<extra></extra>", showlegend=False)]


def bar_traces(entries, lo, hi, active, axis) -> list:
    """NODE_CHART_STYLE = 'bar' — same colours, filled bars instead of stems."""
    xa, ya = axis
    if not entries:
        return []
    xs = [v for v, _ in entries]
    step = min((b - a for a, b in zip(xs, xs[1:])), default=1.0) if len(xs) > 1 else 1.0
    return [go.Bar(x=xs, y=[n for _, n in entries], width=step * 0.7, xaxis=xa, yaxis=ya,
                   marker=dict(color=[viol_color(v, lo, hi, active) for v in xs],
                               line=dict(width=0)), opacity=0.75,
                   hovertemplate="%{x:g} · n=%{y:,}<extra></extra>", showlegend=False)]


def degree_panel(row: dict, thr: dict, axis) -> tuple[list, dict]:
    """Node-degree chart for whichever side NODE_DEGREE_SIDE selects."""
    entry = row["entry"]
    side  = NODE_DEGREE_SIDE
    if entry["src"] == entry["tgt"]:
        side = "src"                       # symmetric subgraph — one degree only
    deg   = row["src_deg"] if side == "src" else row["tgt_deg"]
    t     = thr["src_deg"] if side == "src" else thr["tgt_deg"]
    entries = value_counts(deg)
    traces  = (bar_traces if NODE_CHART_STYLE == "bar" else lollipop_traces)(
        entries, t["lo"], t["hi"], t["active"], axis)
    node_type = entry[side]
    title = f"{'SOURCE' if side == 'src' else 'TARGET'} NODE DEGREE ({entry['label'].upper()})"
    return traces, dict(title=title, x_range=t["full"], thr=t,
                        x_title=f"degree · {node_type}")


def length_panel(row: dict, thr: dict, axis) -> tuple[list, dict]:
    """Edge-length chart: histogram, or the clustered lollipop for discrete lengths."""
    t = thr["length"]
    if row["entry"]["plot"] == "lollipop":
        entries = cluster_by_gap(row["lengths"])
        traces  = lollipop_traces(entries, t["lo"], t["hi"], t["active"], axis)
    else:
        traces  = histogram_traces(row["lengths"], t["full"], t["lo"], t["hi"],
                                   t["active"], LEN_BINS, axis)
    return traces, dict(title=f"EDGE LENGTH ({row['entry']['label'].upper()})",
                        x_range=t["full"], thr=t, x_title="km")


def chart_axis_style(fig, xa: str, ya: str, info: dict, y_max: float):
    """The tool's chart chrome: hairline axes, 8 px dim ticks, no gridlines."""
    xkey = "xaxis" + xa[1:]
    ykey = "yaxis" + ya[1:]
    common = dict(showgrid=False, zeroline=False, showline=True, linecolor=AXIS_LINE,
                  linewidth=1, ticks="outside", ticklen=3, tickcolor=AXIS_LINE,
                  tickfont=dict(size=FONT_XS, color=TEXT_DIM, family=SANS))
    fig.layout[xkey].update(range=list(info["x_range"]), nticks=6,
                            title=dict(text=info["x_title"], font=dict(size=FONT_XS,
                                       color=TEXT_DIM, family=SANS), standoff=2),
                            **common)
    fig.layout[ykey].update(range=[0, y_max * 1.08], nticks=4,
                            title=dict(text="count", font=dict(size=FONT_XS,
                                       color=TEXT_DIM, family=SANS), standoff=2),
                            **common)

def threshold_shapes(xa: str, ya: str, info: dict) -> list[dict]:
    """
    The two dashed threshold markers, as shape dicts.

    Returned rather than added, because the threshold chips swap the whole shape
    list at once and the markers have to travel with the colours they explain.
    xa / ya are trace-style axis references, e.g. "x" and "y3".
    """
    t = info["thr"]
    if not t["active"]:
        return []
    out = []
    for value, color, at_edge in ((t["lo"], VIOL_BELOW, t["lo"] <= t["full"][0]),
                                  (t["hi"], VIOL_ABOVE, t["hi"] >= t["full"][1])):
        if at_edge:
            continue          # a threshold sitting on the range edge flags nothing
        out.append(dict(type="line", x0=value, x1=value, y0=0, y1=1,
                        xref=xa, yref=ya + " domain", opacity=0.9, layer="above",
                        line=dict(color=color, width=1.5, dash="3px,2px")))
    return out

## 6 · Panel geometry, legends, assembly

In [ ]:
# ── Figure 2 · panel geometry, legends, assembly ────────────────────────────
# The 2x4 grid is laid out by hand rather than through make_subplots: the map
# columns are geo subplots and the chart columns are cartesian, and the map crop
# has to match its panel's pixel aspect exactly or Plotly letterboxes the
# projection inside the domain.

def figure_size() -> tuple[int, int]:
    plot_h = 2 * PANEL_H_PX / (1 - V_GAP)
    return FIG_WIDTH_PX, int(round(MARGIN["t"] + MARGIN["b"] + plot_h))

def panel_domains() -> dict:
    """(x0, x1, y0, y1) in paper coordinates for all eight cells, plus panel pixel sizes."""
    fig_w, fig_h = figure_size()
    plot_w = fig_w - MARGIN["l"] - MARGIN["r"]
    plot_h = fig_h - MARGIN["t"] - MARGIN["b"]

    avail = 1 - H_GAP * (len(COL_WIDTHS) - 1)
    total = sum(COL_WIDTHS)
    xs, cursor = [], 0.0
    for w in COL_WIDTHS:
        frac = avail * w / total
        xs.append((round(cursor, 6), round(min(1.0, cursor + frac), 6)))
        cursor += frac + H_GAP

    row_h = (1 - V_GAP) / 2
    ys = [(row_h + V_GAP, 1.0), (0.0, row_h)]     # index 0 = top row

    return dict(x=xs, y=ys, plot_w=plot_w, plot_h=plot_h, fig_w=fig_w, fig_h=fig_h,
                col_px=[(b - a) * plot_w for a, b in xs], row_px=row_h * plot_h)

def auto_aspect(dom: dict) -> float:
    """Map-panel width : height in pixels — the crop aspect that avoids letterboxing."""
    return dom["col_px"][0] / dom["row_px"]


# ── legend cards ────────────────────────────────────────────────────────────
# Redrawn from the tool's own map legends: the same glyphs, the same wording,
# the same semantic colours. Placed as paper-referenced shapes so they float
# over a geo panel the way they float over the map in the UI.

LEGEND_EXPLORE = [
    ("pair",   None,           "grid | mesh"),
    ("arrow",  EDGE_ENCODER,   "grid-to-mesh"),
    ("arrow",  EDGE_DECODER,   "mesh-to-grid"),
    ("line",   EDGE_MESH,      "mesh-to-mesh"),
]
LEGEND_INSPECT = [
    ("circle", VIOL_BELOW,     "under threshold"),
    ("circle", VIOL_ABOVE,     "over threshold"),
    ("note",   None,           "node=degree · edge=length"),
]

def legend_card(fig, dom, anchor, items, width_px=132, corner="bottom-left", pad=9):
    """
    One legend card anchored inside a panel.

    anchor is (x0, x1, y0, y1) in paper coordinates; corner picks which of the
    panel's corners the card hugs. Sizes are given in pixels and converted, so a
    card looks the same whatever the figure's aspect ratio ends up being.
    """
    if not SHOW_LEGENDS:
        return
    s   = LEGEND_SCALE
    W   = width_px * s
    row = 15.0 * s
    fs  = (FONT_XS + 0.5) * s
    H   = pad * 2 * s + row * len(items)

    px2x = lambda p: p / dom["plot_w"]
    px2y = lambda p: p / dom["plot_h"]
    ax0, ax1, ay0, ay1 = anchor
    off = 8 * s
    x0 = ax0 + px2x(off) if "left" in corner else ax1 - px2x(off + W)
    y0 = ay0 + px2y(off) if "bottom" in corner else ay1 - px2y(off + H)
    x1, y1 = x0 + px2x(W), y0 + px2y(H)

    fig.add_shape(type="rect", xref="paper", yref="paper", x0=x0, x1=x1, y0=y0, y1=y1,
                  fillcolor="rgba(255,255,255,0.90)", line=dict(color="rgba(0,0,0,0.12)", width=0.8),
                  layer="above")

    gx    = x0 + px2x(pad * s)            # glyph column, left edge
    gw    = 15.0 * s                      # glyph column width in px
    tx    = gx + px2x(gw + 4 * s)
    dotpx = 5.0 * s

    def dot(cx_px_from_gx, cy, color, symbol):
        cx = gx + px2x(cx_px_from_gx)
        rx, ry = px2x(dotpx / 2), px2y(dotpx / 2)
        kind = "rect" if symbol == "square" else "circle"
        fig.add_shape(type=kind, xref="paper", yref="paper",
                      x0=cx - rx, x1=cx + rx, y0=cy - ry, y1=cy + ry,
                      fillcolor=color, line=dict(color=rgba(TEXT, 200), width=0.6), layer="above")

    def text(x, y, s_, color=TEXT, size=None, anchor_="left"):
        fig.add_annotation(xref="paper", yref="paper", x=x, y=y, text=s_, showarrow=False,
                           xanchor=anchor_, yanchor="middle",
                           font=dict(size=size or fs, color=color, family=SANS))

    for i, (kind, color, label) in enumerate(items):
        cy = y1 - px2y(pad * s + row * (i + 0.5))
        if kind == "pair":
            dot(dotpx / 2, cy, NODE_GRID, "square")
            text(gx + px2x(dotpx + 3 * s), cy, "grid")
            dot(gw + 24 * s, cy, NODE_MESH, "circle")
            text(gx + px2x(gw + 24 * s + dotpx / 2 + 3 * s), cy, "mesh")
            continue
        if kind == "arrow":
            # Annotation arrows measure their tail in pixels, so the glyph keeps its
            # length whatever the figure's aspect ratio is.
            fig.add_annotation(xref="paper", yref="paper", axref="pixel", ayref="pixel",
                               x=gx + px2x(gw), y=cy, ax=-gw, ay=0, text="", showarrow=True,
                               arrowhead=2, arrowsize=1.1, arrowwidth=1.2 * s, arrowcolor=color)
        elif kind == "line":
            fig.add_shape(type="line", xref="paper", yref="paper",
                          x0=gx, x1=gx + px2x(gw), y0=cy, y1=cy,
                          line=dict(color=color, width=1.4 * s), layer="above")
        elif kind == "circle":
            dot(dotpx / 2 + 2 * s, cy, color, "circle")
        elif kind == "note":
            text(gx, cy, label, color=TEXT_MUTED, size=fs - 0.5)
            continue
        text(tx, cy, label)


# ── assembly ────────────────────────────────────────────────────────────────

def _sfx(n: int) -> str:
    """Plotly names the first subplot of a kind without a number: geo, xaxis, x."""
    return "" if n == 1 else str(n)


def panel_label(fig, dom, anchor, text, dy_px=8):
    fig.add_annotation(xref="paper", yref="paper", x=anchor[0], y=anchor[3] + dy_px / dom["plot_h"],
                       text=text, showarrow=False, xanchor="left", yanchor="bottom",
                       font=dict(size=FONT_XS, color=TEXT_MUTED, family=SANS))

def row_caption(fig, dom, anchor, text, dy_px=24):
    fig.add_annotation(xref="paper", yref="paper", x=anchor[0], y=anchor[3] + dy_px / dom["plot_h"],
                       text=f"<b>{text}</b>", showarrow=False, xanchor="left", yanchor="bottom",
                       font=dict(size=FONT_S + 1, color=TEXT, family=SANS))


def threshold_chips(fig, dom, presets, active_idx, vis_by_preset, shapes_by_preset):
    """
    The chip row above the figure.

    Every preset's Inspect panels and charts are drawn once and then hidden, so a
    chip is a pure visibility swap plus the matching threshold markers — no
    recomputation, and the whole thing survives export to a standalone HTML file.
    """
    buttons = [
        dict(label=f"  {label}  ", method="update",
             args=[{"visible": vis_by_preset[key]}, {"shapes": shapes_by_preset[key]}])
        for key, label in presets
    ]
    y = 1.0 + CHIP_ROW_Y_PX / dom["plot_h"]     # clear of the row captions below
    fig.update_layout(updatemenus=[dict(
        type="buttons", direction="right", showactive=True, active=active_idx,
        buttons=buttons, x=0.052, xanchor="left", y=y, yanchor="bottom",
        pad=dict(l=0, r=0, t=1, b=1), borderwidth=0.8,
        bgcolor="rgba(255,255,255,0.94)", bordercolor="rgba(0,0,0,0.12)",
        font=dict(size=FONT_XS, color=TEXT, family=SANS),
    )])
    fig.add_annotation(xref="paper", yref="paper", x=0.0, y=y, xanchor="left", yanchor="bottom",
                       yshift=7, text="THRESHOLD", showarrow=False,
                       font=dict(size=FONT_XS - 1, color=TEXT_DIM, family=SANS))


def build_figure(rows_spec=None, presets=None, active=None, custom=None) -> go.Figure:
    """
    The whole 2x4 figure: Explore | Inspect | node degree | edge length, per graph.

    presets  list of (key, label) chips, or a single preset key for a figure with
             no chips at all — which is what a static export wants.
    active   the preset selected on open; defaults to THRESHOLD_PRESET.
    """
    rows_spec = rows_spec or ROWS
    if isinstance(presets, str):
        presets, chips = [(presets, presets)], False
    else:
        presets, chips = (presets or THRESHOLD_CHIPS), True
    keys   = [k for k, _ in presets]
    active = active or THRESHOLD_PRESET
    if active not in keys:
        keys, presets = [active] + keys, [(active, active)] + list(presets)
    active_idx = keys.index(active)

    dom    = panel_domains()
    fig    = go.Figure()
    aspect = auto_aspect(dom)
    trace_preset: list[str | None] = []          # None = shown under every chip
    shapes_by_preset: dict[str, list] = {k: [] for k in keys}

    def add(trace, preset=None):
        fig.add_trace(trace)
        trace_preset.append(preset)

    for r, spec in enumerate(rows_spec):
        crop = dict(spec.get("crop") or CROP)
        if crop.get("aspect") in (None, "auto"):
            crop["aspect"] = aspect
        row = build_row(dict(spec, crop=crop))

        e_sfx, i_sfx = _sfx(2 * r + 1), _sfx(2 * r + 2)
        geo_explore, geo_inspect = f"geo{e_sfx}", f"geo{i_sfx}"
        xa,  ya  = f"x{e_sfx}", f"y{e_sfx}"
        xa2, ya2 = f"x{i_sfx}", f"y{i_sfx}"
        anchors = [(dom["x"][c][0], dom["x"][c][1], dom["y"][r][0], dom["y"][r][1])
                   for c in range(4)]

        for t in explore_traces(row, geo_explore, dom["col_px"][0]):
            add(t)

        deg_info = len_info = None
        deg_max = len_max = 1.0
        for key in keys:
            thr = row_thresholds(row, key, custom)
            for t in inspect_traces(row, thr, geo_inspect, dom["col_px"][1]):
                add(t, key)
            deg_traces, deg_info = degree_panel(row, thr, (xa, ya))
            len_traces, len_info = length_panel(row, thr, (xa2, ya2))
            for t in deg_traces + len_traces:
                add(t, key)
            deg_max = max([deg_max] + [float(np.nanmax(t.y)) for t in deg_traces if len(t.y)])
            len_max = max([len_max] + [float(np.nanmax(t.y)) for t in len_traces if len(t.y)])
            shapes_by_preset[key] += threshold_shapes(xa, ya, deg_info)
            shapes_by_preset[key] += threshold_shapes(xa2, ya2, len_info)

        g1 = geo_layout(row); g1["domain"] = dict(x=list(dom["x"][0]), y=list(dom["y"][r]))
        g2 = geo_layout(row); g2["domain"] = dict(x=list(dom["x"][1]), y=list(dom["y"][r]))
        fig.update_layout({geo_explore: g1, geo_inspect: g2})

        for (ax, ay), col in (((xa, ya), 2), ((xa2, ya2), 3)):
            fig.update_layout({
                f"xaxis{ax[1:]}": dict(domain=list(dom["x"][col]), anchor=ay),
                f"yaxis{ay[1:]}": dict(domain=list(dom["y"][r]),  anchor=ax),
            })
        chart_axis_style(fig, xa, ya, deg_info, deg_max)
        chart_axis_style(fig, xa2, ya2, len_info, len_max)

        counts = " · ".join(f"{k} {v:,}" for k, v in row["meta"]["node_counts"].items())
        row_caption(fig, dom, anchors[0], row["label"])
        panel_label(fig, dom, anchors[0], f"EXPLORE — {counts}")
        panel_label(fig, dom, anchors[1], f"INSPECT — {row['entry']['label']}")
        panel_label(fig, dom, anchors[2], deg_info["title"])
        panel_label(fig, dom, anchors[3], len_info["title"])

        if r == 0:
            legend_card(fig, dom, anchors[0], LEGEND_EXPLORE, width_px=136, corner="bottom-left")
            legend_card(fig, dom, anchors[1], LEGEND_INSPECT, width_px=124, corner="bottom-left")

    # Legend shapes were added straight to the layout; they belong to every chip.
    base_shapes = [sh.to_plotly_json() for sh in fig.layout.shapes]
    shapes_by_preset = {k: base_shapes + v for k, v in shapes_by_preset.items()}
    vis_by_preset = {k: [tp is None or tp == k for tp in trace_preset] for k in keys}
    for t, vis in zip(fig.data, vis_by_preset[active]):
        t.visible = vis
    fig.update_layout(shapes=shapes_by_preset[active])

    fig_w, fig_h = figure_size()
    fig.update_layout(width=fig_w, height=fig_h, margin=MARGIN,
                      paper_bgcolor="white", plot_bgcolor="white",
                      font=dict(family=SANS, size=FONT_S, color=TEXT),
                      showlegend=False, bargap=0, hovermode="closest")
    if chips and len(presets) > 1:
        threshold_chips(fig, dom, presets, active_idx, vis_by_preset, shapes_by_preset)
    return fig

## 7 · Build itThe chip row at the top of the figure sets the threshold. The widgets below choosethe graphs, the subgraph each row is inspected on, and the crop.

In [ ]:
# ── Figure 2 · interactive controls ─────────────────────────────────────────
# Two kinds of control, split by what they cost.
#
#   Threshold  lives inside the figure as the chip row, because a threshold only
#              recolours marks that are already drawn — so every preset is drawn
#              once, up front, and a chip just swaps which one is visible. That
#              also means the chips keep working in an exported HTML file.
#   Everything else (which graph, which subgraph, where the crop sits) changes
#              what has to be loaded and cropped, so it lives in widgets that
#              rebuild the figure.

import ipywidgets as W
from IPython.display import display, clear_output

_GRAPHS = available_graphs()

def _subgraph_options(graph_name: str) -> list[str]:
    try:
        return sorted(open_graph(graph_name)["registry"].keys())
    except Exception as e:                       # a folder that will not load
        print(f"{graph_name}: {e}")
        return ["g2m"]

def _row_controls(default: dict, n: int) -> dict:
    graph = W.Dropdown(options=_GRAPHS, value=default["graph"],
                       description=f"row {n} graph", style={"description_width": "88px"},
                       layout=W.Layout(width="330px"))
    sub   = W.Dropdown(options=_subgraph_options(default["graph"]), value=default["subgraph"],
                       description="subgraph", style={"description_width": "72px"},
                       layout=W.Layout(width="220px"))
    label = W.Text(value=default.get("label") or default["graph"], description="label",
                   style={"description_width": "48px"}, layout=W.Layout(width="260px"))

    def on_graph(change):
        opts = _subgraph_options(change["new"])
        sub.options = opts
        sub.value = "g2m" if "g2m" in opts else opts[0]
        if label.value in _GRAPHS:
            label.value = change["new"]
    graph.observe(on_graph, names="value")
    return dict(graph=graph, sub=sub, label=label)

_r1 = _row_controls(ROWS[0], 1)
_r2 = _row_controls(ROWS[1], 2)

_lat  = W.FloatText(value=CROP["center"][0], step=1.0, description="centre lat",
                    style={"description_width": "72px"}, layout=W.Layout(width="180px"))
_lon  = W.FloatText(value=CROP["center"][1], step=1.0, description="lon",
                    style={"description_width": "32px"}, layout=W.Layout(width="140px"))
_hgt  = W.FloatSlider(value=CROP["height"], min=1.0, max=170.0, step=0.5, readout_format=".1f",
                      description="crop °lat", style={"description_width": "72px"},
                      continuous_update=False, layout=W.Layout(width="330px"))
_nsz  = W.FloatSlider(value=NODE_SIZE_SCALE, min=0.3, max=4.0, step=0.1,
                      description="node size", style={"description_width": "72px"},
                      continuous_update=False, layout=W.Layout(width="260px"))
_ewd  = W.FloatSlider(value=EDGE_WIDTH_SCALE, min=0.1, max=2.0, step=0.05,
                      description="edge width", style={"description_width": "72px"},
                      continuous_update=False, layout=W.Layout(width="260px"))
_bmap = W.FloatSlider(value=BASEMAP_ALPHA, min=0.0, max=1.0, step=0.05,
                      description="basemap", style={"description_width": "72px"},
                      continuous_update=False, layout=W.Layout(width="260px"))
_chip = W.Dropdown(options=[(lbl, k) for k, lbl in THRESHOLD_CHIPS], value=THRESHOLD_PRESET,
                   description="chip on open", style={"description_width": "88px"},
                   layout=W.Layout(width="240px"))
_arrw = W.Checkbox(value=EXPLORE_ARROWS, description="edge arrowheads", indent=False,
                   layout=W.Layout(width="170px"))
_out  = W.Output()

FIG = None      # the last figure built — cell 8 exports this one

def current_rows() -> list[dict]:
    return [dict(graph=c["graph"].value, subgraph=c["sub"].value, label=c["label"].value, crop=None)
            for c in (_r1, _r2)]

def render(*_):
    global FIG, NODE_SIZE_SCALE, EDGE_WIDTH_SCALE, EXPLORE_ARROWS, BASEMAP_ALPHA
    NODE_SIZE_SCALE, EDGE_WIDTH_SCALE, EXPLORE_ARROWS = _nsz.value, _ewd.value, _arrw.value
    BASEMAP_ALPHA = _bmap.value
    CROP.update(center=(_lat.value, _lon.value), height=_hgt.value, aspect="auto")
    with _out:
        clear_output(wait=True)
        try:
            FIG = build_figure(current_rows(), active=_chip.value)
            display(FIG)
        except Exception as e:
            print(f"{type(e).__name__}: {e}")

for w in (_r1["graph"], _r1["sub"], _r1["label"], _r2["graph"], _r2["sub"], _r2["label"],
          _lat, _lon, _hgt, _nsz, _ewd, _bmap, _chip, _arrw):
    w.observe(render, names="value")

display(W.VBox([
    W.HBox([_r1["graph"], _r1["sub"], _r1["label"]]),
    W.HBox([_r2["graph"], _r2["sub"], _r2["label"]]),
    W.HBox([_lat, _lon, _hgt, _chip]),
    W.HBox([_nsz, _ewd, _bmap, _arrw]),
]))
display(_out)
render()

## 8 · Export

In [ ]:
# ── Figure 2 · export ───────────────────────────────────────────────────────
# The chips are a building aid, not part of the figure. A static export drops
# them and bakes in whichever preset is selected, so what lands in the paper is
# exactly what is on screen minus the buttons.

OUT_STEM  = ROOT / "notebooks" / "figure2_explore_inspect"
EXPORT_AS = ["pdf", "png"]        # pdf for the paper, png for slides/drafts
EXPORT_SCALE = 2                  # png only; pdf is vector

static = build_figure(current_rows(), presets=_chip.value)
for fmt in EXPORT_AS:
    path = OUT_STEM.with_suffix(f".{fmt}")
    static.write_image(path, scale=EXPORT_SCALE if fmt == "png" else 1)
    print("wrote", path)

# The interactive version, chips and all, as a standalone file to share.
html = OUT_STEM.with_suffix(".html")
build_figure(current_rows(), active=_chip.value).write_html(html, include_plotlyjs="cdn")
print("wrote", html)